In [1]:
%load_ext autoreload
%autoreload 2

In [23]:
import pandas as pd
import os
import json
import dotenv

from citation_extractor import extract_echr_citations
from llm import get_completion
from prompts import get_summarizer_prompt, get_analysis_prompt
from utils import load_json, save_json, normalize, find_closest_match


dotenv.load_dotenv()

True

In [108]:
data_path = '/Users/ahmed/Desktop/msc-24/ECHR_v2/echr_processed/'

df1_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_1.csv'
df2_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_2.csv'
df3_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_3.csv'
df4_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_4.csv'

df1 = pd.read_csv(df1_path)['file_path'].to_list()
df2 = pd.read_csv(df2_path)['file_path'].to_list()
df3 = pd.read_csv(df3_path)['file_path'].to_list()
df4 = pd.read_csv(df4_path)['file_path'].to_list()

id_to_name = load_json('/Users/ahmed/Desktop/msc-24/ECHR_v2/id_to_name.json')
name_to_id = {v: k for k, v in id_to_name.items()}
cases_names = list(id_to_name.values())

In [131]:
case_path = data_path + df1[34]
case = load_json(case_path)
text_list = case['law']

print(case['itemid'])

results = extract_echr_citations(text_list)
results_ids = []
results_paths = []
results_names = []

for item in results:
    cited_case_name = item['citation']['case_name']
    cited_case_name = normalize(cited_case_name)
    match_name = find_closest_match(cited_case_name, cases_names)
    cited_case_id = None
    try:
        cited_case_id = name_to_id[match_name]
    except:
        cited_case_id = None
    if cited_case_id:
        results_names.append(match_name)
        results_ids.append(cited_case_id)
        cited_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed/' + cited_case_id + '.json'
        results_paths.append(cited_path)


print(len(results))
print(len(results_ids))
print(len(results_names))

001-205536
57
48
48


In [132]:
importances = []
for path in results_paths:
    case = load_json(path)
    importance = case['importance']
    importances.append(importance)

In [133]:
for i,j,k,l in zip(results_ids, results_names, results_paths, importances):
    print(i,j,k,l)

001-99015 GAFGEN V GERMANY /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-99015.json 1
001-115621 ELMASRI V THE FORMER YUGOSLAV REPUBLIC OF MACEDONIA /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-115621.json 1
001-157670 BOUYID V BELGIUM /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-157670.json 1
001-157277 KHLAIFIA AND OTHERS V ITALY /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-157277.json 3
001-152730 MURSIC V CROATIA /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-152730.json 3
001-169662 PAPOSHVILI V BELGIUM /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-169662.json 1
001-194307 NICOLAE VIRGILIU TANASE V ROMANIA /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-194307.json 1
001-60822 A V THE UNITED KINGDOM /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-60822.json 1
001-140235 OKEEFFE V IRELAND /Users/ahmed/Desktop/msc-24/ECHR/echr-processed/001-140235.json 1
001-58257 OSMAN V THE UNITED KINGDOM /Users/ahmed/Desktop/msc-24/ECHR/echr-processed

In [96]:
def Summarize(case_path):
    data_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed/'
    case_path = data_path + df1[0]
    case = load_json(case_path)
    case_text = 'FACTS: ' + case['facts']  + '\n  LAW:' + case['law']
    summarize_prompt = get_summarizer_prompt(case_text)
    completion = get_completion(summarize_prompt)
    return completion

def Analyze(case_a_summary, case_b_summary):
    analyses_prompt = get_analysis_prompt(case_a_summary, case_b_summary)
    completion = get_completion(analyses_prompt)
    return completion

In [97]:
main_case_summary = Summarize(case_path)

In [98]:
print(main_case_summary)

The case involves an applicant, born in 1973, serving a life sentence in Ukraine after being convicted in Hungary for a double murder. The Hungarian courts sentenced him to life imprisonment with the possibility of parole after twenty years. Following his transfer to Ukraine in 2007, the Ukrainian courts recognized the Hungarian sentence but classified it as irreducible under Ukrainian law, effectively denying him the possibility of parole.

The applicant's requests for parole were repeatedly denied by Ukrainian courts, which cited the lack of a legal framework for parole for life prisoners. The applicant argued that this situation violated Article 3 of the European Convention on Human Rights (ECHR), which prohibits inhuman or degrading treatment, and Article 7, which protects against retroactive penalties.

The European Court of Human Rights (ECHR) found the complaint under Article 3 admissible, noting that an irreducible life sentence is incompatible with the Convention's requirement

In [99]:
summaries = []
analyses = []

In [100]:
for path in results_paths[3]:
    #print(path)
    case = load_json(path)
    summary = Summarize(path)
    summaries.append(summary)
    print('summarized case:', case['itemid'])
    break


summarized case: 001-122664


In [101]:
summaries[0]

"The case involves an applicant, born in 1973, serving a life sentence in Ukraine after being convicted in Hungary for a double murder. The applicant was arrested in Hungary on December 8, 1999, and sentenced to life imprisonment on March 21, 2002, with the possibility of parole after twenty years. Following unsuccessful attempts to transfer him to Ukraine with his consent, he was transferred on May 8, 2007, under the Additional Protocol to the Convention on the Transfer of Sentenced Persons.\n\nUkrainian courts recognized the Hungarian sentence but classified it under Ukrainian law, which resulted in the applicant being deemed to have an irreducible life sentence, effectively removing the possibility of parole. The applicant made multiple requests for parole, all of which were denied by Ukrainian courts, citing the lack of a legal framework for parole for life prisoners.\n\nThe applicant alleged violations of Article 3 (prohibition of inhuman or degrading treatment) and Article 7 (no 

In [27]:
for idx, summary in enumerate(summaries):
    analysis = Analyze(main_case_summary, summary)
    analyses.append(analysis)
    print('analyzed main case with case number', idx + 1)

analyzed main case with case number 1
analyzed main case with case number 2
analyzed main case with case number 3
analyzed main case with case number 4
analyzed main case with case number 5
analyzed main case with case number 6
analyzed main case with case number 7
analyzed main case with case number 8
analyzed main case with case number 9
analyzed main case with case number 10
analyzed main case with case number 11
analyzed main case with case number 12
analyzed main case with case number 13
analyzed main case with case number 14
analyzed main case with case number 15
analyzed main case with case number 16


In [30]:
print(analyses[1])

In analyzing the connection between Case A and Case B, we can identify several key legal principles and facts from Case B that Case A relies on, as well as the manner in which Case B is utilized in Case A.

### Legal Principles and Facts from Case B

1. **Article 5 Violations**: Both cases center around violations of Article 5 of the European Convention on Human Rights, which protects the right to liberty and security. Case B specifically addresses issues of unlawful detention, lack of judicial review, and absence of proper arrest records, which are critical elements that underpin the Court's findings in Case A.

2. **Lawfulness of Detention**: Case B emphasizes that compliance with national law is essential for lawful detention. This principle is echoed in Case A, where the Court reiterates its authority to review adherence to domestic law in the context of the applicants' complaints about unlawful detention.

3. **Judicial Review and Access to Rights**: In Case B, the Court highlight

In [102]:
def get_analysis_prompt_exp(case_A: str, case_B: str) -> str:
    return f"""
You are a specialist in European Court of Human Rights (ECHR) jurisprudence.    

Case A cites Case B. Your task is to analyze this citation comprehensively. Focus on these aspects:
1. **Reason for Citation**: Why does Case A cite Case B? What specific arguments, legal principles, or facts from Case B are relevant to Case A's reasoning?
2. **Connection**: How are the two cases connected in terms of legal principles, facts, or procedural elements? Provide specific examples.
3. **Usage of Case B**: How does Case A use Case B (e.g., as precedent, analogy, or to contrast)? Explain this in detail.
4. **Influence**: How do Case B’s reasoning and outcome shape or support the arguments in Case A?
5. **Critical Context**: Highlight any differences or nuances in how the principles or facts are interpreted in the two cases.
6. what is novel in case A compared to case B?

Then come up with the final answer in the following format:
Common Articles:
Issue of case A:
Issue of case B:
Relevance: 
Importance of citation: (Primary, Secondary, Contextual)
How cited: (quote directly, cited as precedent, cited as analogy, cited as contrast)


Here are the summaries:

Case A Reasoning Summary: 
{case_A}

Case B Reasoning Summary: 
{case_B}

Provide a structured and detailed analysis focusing on the connection, reasoning, and practical implications.
"""

analyses_prompt = get_analysis_prompt_exp(main_case_summary, summary[0])
completion = get_completion(analyses_prompt)


In [103]:
print(completion)

To provide a comprehensive analysis of the citation from Case A to Case B, we will break down the aspects as requested.

### Common Articles:
- Article 3: Prohibition of inhuman or degrading treatment.
- Article 7: No punishment without law (prohibition of retroactive penalties).

### Issue of Case A:
The issue in Case A revolves around the applicant's life sentence being classified as irreducible under Ukrainian law, which effectively denied him the possibility of parole, leading to claims of violations of Article 3 and Article 7 of the ECHR.

### Issue of Case B:
The issue in Case B likely pertains to the treatment of life sentences and the conditions under which they can be deemed compatible with the ECHR, particularly focusing on the principles of proportionality and the right to a review of sentences.

### Relevance:
Case B is relevant to Case A as it establishes important legal principles regarding the treatment of life sentences and the necessity for a mechanism to review such s